# Part 8: Machine Learning for Data Scientists
Machine Learning (ML) is a subset of AI where systems learn patterns from data rather than being explicitly programmed.

## 1. Types of ML Algorithms

* **Supervised Learning:** The model learns from labeled data (Input + Correct Answer). 
    * *Examples:* Predicting house prices (Regression), Email spam detection (Classification).
* **Unsupervised Learning:** The model finds hidden patterns in unlabeled data.
    * *Examples:* Customer segmentation (Clustering), Reducing data dimensions (PCA).
* **Reinforcement Learning:** The model learns by trial and error through rewards/penalties.
    * *Example:* Training an AI to play chess or navigate a robot.

In [1]:
# Installation (Run this in your VS Code terminal or uncomment here)
# !pip install scikit-learn tensorflow joblib

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.datasets import load_iris
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score

# 1. Our First ML Model: Training on the famous Iris Dataset
# The goal is to classify the species of a flower based on its petal/sepal dimensions.
iris = load_iris()
X, y = iris.data, iris.target

# Split data into Training (80%) and Testing (20%) sets
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Initialize and Train the model
model = RandomForestClassifier(n_estimators=100)
model.fit(X_train, y_train)

# Make predictions and check accuracy
predictions = model.predict(X_test)
print(f"Iris Model Accuracy: {accuracy_score(y_test, predictions):.2%}")

Iris Model Accuracy: 100.00%


# Part 9: Practical ML using Scikit-learn
Real-world data is rarely as clean as the Iris dataset. It contains missing values, text categories, and different scales. 

## The House Price Prediction Project
When planning to build a house in an area like Daudnagar, Lucknow, having a predictive model to estimate property values based on square footage, bedroom count, and location features is incredibly practical. We will build an end-to-end **Pipeline** to handle this.

In [2]:
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error, r2_score
import joblib

# Simulated Real Estate Data
data = {
    'Area_SqFt': [1200, 1500, np.nan, 2000, 850, 2500],
    'Bedrooms': [2, 3, 3, 4, 2, 4],
    'Location_Type': ['Urban', 'Suburban', 'Urban', 'Rural', 'Urban', 'Suburban'],
    'Price_Lakhs': [45.5, 60.0, 55.0, 75.0, 35.0, 90.0] # Target Variable
}
df_housing = pd.DataFrame(data)

# Separate Features (X) and Target (y)
X = df_housing.drop('Price_Lakhs', axis=1)
y = df_housing['Price_Lakhs']

# 1. Preprocessing Pipelines
# For numbers: Fill missing values with the median, then scale them
numeric_features = ['Area_SqFt', 'Bedrooms']
numeric_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler())
])

# For text/categories: Fill missing with 'missing', then One-Hot Encode (turn into binary columns)
categorical_features = ['Location_Type']
categorical_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='constant', fill_value='missing')),
    ('onehot', OneHotEncoder(handle_unknown='ignore'))
])

# Combine them using ColumnTransformer
preprocessor = ColumnTransformer(
    transformers=[
        ('num', numeric_transformer, numeric_features),
        ('cat', categorical_transformer, categorical_features)
    ])

# 2. Build the Final Pipeline (Preprocessing + Model)
ml_pipeline = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('regressor', LinearRegression())
])

# 3. Train the Model
ml_pipeline.fit(X, y)

# 4. Evaluate Error Metrics
y_pred = ml_pipeline.predict(X)
print(f"Mean Squared Error (MSE): {mean_squared_error(y, y_pred):.2f}")
print(f"R-squared (Accuracy): {r2_score(y, y_pred):.2%}")

# 5. Model Persistence: Save the trained pipeline to disk
joblib.dump(ml_pipeline, 'house_price_model.pkl')
print("\n✅ Model saved as 'house_price_model.pkl'")

Mean Squared Error (MSE): 0.01
R-squared (Accuracy): 100.00%

✅ Model saved as 'house_price_model.pkl'


# Part 10: Deep Learning & Neural Networks
Deep Learning is a specialized branch of ML that uses **Artificial Neural Networks** inspired by the human brain.

## 1. Introduction & Terminology

* **Perceptron:** The simplest neural network (a single neuron). It takes inputs, multiplies them by **Weights**, adds a **Bias**, and passes the result through an **Activation Function**.
* **Hidden Layers:** Layers of neurons between the input and output. Adding many hidden layers makes the network "Deep".
* **Frameworks:** **PyTorch** (favored in research), **TensorFlow** (Google's production standard), and **Keras** (a user-friendly interface built into TensorFlow).

## 2. Training a Neural Network on MNIST
The MNIST dataset is the "Hello World" of Deep Learning. It consists of 70,000 images of handwritten digits (0-9). We will build a multi-layer Neural Network to recognize them.

In [3]:
%pip install tensorflow

Note: you may need to restart the kernel to use updated packages.


In [4]:
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Flatten
from sklearn.linear_model import Perceptron

# --- Part A: The Scikit-Learn Perceptron ---
# A basic single-neuron classifier
clf = Perceptron(max_iter=1000, random_state=42)
# (In practice, you would fit this like: clf.fit(X_train, y_train))
print("Scikit-learn Perceptron initialized.\n")


# --- Part B: Deep Learning with TensorFlow/Keras ---
print(f"TensorFlow Version: {tf.__version__}")

# 1. Load the MNIST Dataset
mnist = tf.keras.datasets.mnist
(x_train, y_train), (x_test, y_test) = mnist.load_data()

# Normalize pixel values to be between 0 and 1
x_train, x_test = x_train / 255.0, x_test / 255.0

# 2. Build the Neural Network Architecture
model = Sequential([
    Flatten(input_shape=(28, 28)),          # Input Layer: Flattens the 28x28 image into a 1D array
    Dense(128, activation='relu'),          # Hidden Layer: 128 neurons using ReLU activation
    Dense(10, activation='softmax')         # Output Layer: 10 neurons (for digits 0-9) returning probabilities
])

# 3. Compile the Model (Specify Optimizer and Loss Function)
model.compile(optimizer='adam',
              loss='sparse_categorical_crossentropy',
              metrics=['accuracy'])

# 4. Train the Neural Network (Running for 3 epochs to save time)
print("\nTraining Neural Network on MNIST...")
model.fit(x_train, y_train, epochs=3, validation_split=0.1)

# 5. Evaluate Accuracy
test_loss, test_acc = model.evaluate(x_test,  y_test, verbose=2)
print(f"\nNeural Network Test Accuracy: {test_acc:.2%}")

Scikit-learn Perceptron initialized.

TensorFlow Version: 2.21.0


c:\Users\shiva\.conda\envs\datasci\lib\site-packages\keras\src\layers\reshaping\flatten.py:37: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)



Training Neural Network on MNIST...
Epoch 1/3
1688/1688 ━━━━━━━━━━━━━━━━━━━━ 5s 3ms/step - accuracy: 0.9223 - loss: 0.2733 - val_accuracy: 0.9642 - val_loss: 0.1294
Epoch 2/3
1688/1688 ━━━━━━━━━━━━━━━━━━━━ 5s 3ms/step - accuracy: 0.9647 - loss: 0.1223 - val_accuracy: 0.9730 - val_loss: 0.0955
Epoch 3/3
1688/1688 ━━━━━━━━━━━━━━━━━━━━ 5s 3ms/step - accuracy: 0.9751 - loss: 0.0834 - val_accuracy: 0.9750 - val_loss: 0.0870
313/313 - 0s - 1ms/step - accuracy: 0.9740 - loss: 0.0879

Neural Network Test Accuracy: 97.40%
